In [1]:
from webull_bridge import WebullBridge, WebullCredentials

# โหลด credentials จาก .env
creds = WebullCredentials.from_env()

print("Region :", creds.region_id)
print("Sandbox:", creds.sandbox)
print("Account:", creds.account_id)

# สร้าง Webull Bridge
bridge = WebullBridge(creds)

print("\n✅ Webull connected successfully!")

Region : us
Sandbox: True
Account: 5AJAB8N6QSG63RDH4A348K2VKA
25900 2026-09-30 10:34:04,761 webull.core.http.initializer.client_initializer INFO _check_token_enable result is False

✅ Webull connected successfully!


In [2]:
import os

# 1. โหลด Credentials และเชื่อมต่อ Bridge
creds = WebullCredentials.from_env()
bridge = WebullBridge(creds)

# 2. ดึงข้อมูลสรุปบัญชีจาก Sandbox
summary = bridge.get_account_summary()

print("=== RAW RESPONSE จาก WEBULL SANDBOX ===")
print(summary)
print("========================================\n")

# 3. ตรวจสอบและดึงข้อมูลอย่างปลอดภัย
if isinstance(summary, dict):
    # กรณีข้อมูลซ้อนอยู่ใน key 'data' หรือเป็น dict ชั้นเดียว
    data = summary.get("data", summary) if isinstance(summary.get("data"), dict) else summary

    # ค้นหาชื่อฟีลด์ที่เป็นไปได้ตามโครงสร้าง Webull OpenAPI
    cash = data.get("total_cash_balance") or data.get("usable_cash") or data.get("cash_balance") or 0
    net_val = data.get("total_net_liquidation_value") or data.get("net_liquidation_value") or 0
    mkt_val = data.get("total_market_value") or data.get("market_value") or 0

    try:
        cash_num = float(cash)
        net_num = float(net_val)
        mkt_num = float(mkt_val)

        print("==========================================")
        print("      สรุปยอดเงินใน Sandbox (Paper)       ")
        print("==========================================")
        print(f"💵 เงินสดคงเหลือ (Cash Balance):    ${cash_num:,.2f}")
        print(f"📈 มูลค่าหุ้นที่ถืออยู่ (Market Value): ${mkt_num:,.2f}")
        print(f"💰 มูลค่าพอร์ตรวม (Net Value):       ${net_num:,.2f}")
        print("==========================================")
    except (ValueError, TypeError) as e:
        print(f"⚠️ แปลงตัวเลขไม่สำเร็จ กรุณาเช็กชื่อฟีลด์จาก RAW RESPONSE ด้านบน")

25900 2026-09-30 10:34:07,707 webull.core.http.initializer.client_initializer INFO _check_token_enable result is False
25900 2026-09-30 10:34:07,707 webull.core.http.initializer.client_initializer INFO _check_token_enable result is False
=== RAW RESPONSE จาก WEBULL SANDBOX ===
{'total_asset_currency': 'USD', 'total_net_liquidation_value': '1000000.00', 'total_market_value': '0.00', 'total_cash_balance': '1000000.00', 'total_unrealized_profit_loss': '0.00', 'total_day_profit_loss': '0.00', 'day_trades_left': 'UNLIMITED', 'maintenance_margin': '0.00', 'open_margin_calls': [], 'account_currency_assets': [{'currency': 'USD', 'net_liquidation_value': '1000000.00', 'market_value': '0.00', 'cash_balance': '1000000.00', 'option_buying_power': '1000000.00', 'day_buying_power': '4000000', 'overnight_buying_power': '2000000.00', 'night_trading_buying_power': '1000000.00', 'unrealized_profit_loss': '0.00', 'day_profit_loss': '0.00'}]}

      สรุปยอดเงินใน Sandbox (Paper)       
💵 เงินสดคงเหลือ (Ca

In [3]:
from datetime import datetime


# ============================================================
# FORMAT FUNCTIONS
# ============================================================

def money(x):
    try:
        return f"${float(x):,.2f}"
    except:
        return "-"


def signed_money(x):
    try:
        value = float(x)

        if value >= 0:
            return f"+${value:,.2f}"
        else:
            return f"-${abs(value):,.2f}"

    except:
        return "-"


def line(char="-", width=78):
    print(char * width)


# ============================================================
# GET DATA
# ============================================================

# get_account_summary() คืน balance โดยตรง
balance = bridge.get_account_summary()

positions = bridge.get_positions()

updated = datetime.now().strftime("%Y-%m-%d %H:%M:%S")


# ============================================================
# ACCOUNT INFORMATION
# ============================================================

currency = balance["total_asset_currency"]

# จาก credentials ของเรา
environment = "SANDBOX / PAPER" if creds.sandbox else "LIVE"


# ============================================================
# HEADER
# ============================================================

print()
line("=")
print("WEBULL PORTFOLIO DASHBOARD".center(78))
line("=")

print()

print(f"⏱️ Updated      : {updated}")
print(f"🌎 Environment  : {environment}")
print(f"💵 Currency     : {currency}")


# ============================================================
# ACCOUNT SUMMARY
# ============================================================

print()

line()
print("ACCOUNT SUMMARY".center(78))
line()

print(
    f"💰 Net Liquidation Value : "
    f"{money(balance['total_net_liquidation_value'])}"
)

print(
    f"💵 Cash Balance          : "
    f"{money(balance['total_cash_balance'])}"
)

print(
    f"📊 Market Value          : "
    f"{money(balance['total_market_value'])}"
)

print(
    f"📈 Unrealized P/L        : "
    f"{signed_money(balance['total_unrealized_profit_loss'])}"
)

print(
    f"📅 Day P/L               : "
    f"{signed_money(balance['total_day_profit_loss'])}"
)

print(
    f"🛡️ Maintenance Margin    : "
    f"{money(balance['maintenance_margin'])}"
)


# ============================================================
# PORTFOLIO
# ============================================================

print()

line()
print("PORTFOLIO".center(78))
line()

print(
    f"{'Symbol':<10}"
    f"{'Qty':>8}"
    f"{'Price':>14}"
    f"{'Market Value':>18}"
    f"{'P/L':>14}"
)

line()

for p in positions:

    print(
        f"{p['symbol']:<10}"
        f"{int(float(p['quantity'])):>8,}"
        f"{money(p['last_price']):>14}"
        f"{money(p['market_value']):>18}"
        f"{signed_money(p['unrealized_profit_loss']):>14}"
    )


# ============================================================
# STATUS
# ============================================================

print()

line()
print("STATUS".center(78))
line()

open_orders = bridge.list_open_orders()

print(f"🟢 Open Orders : {len(open_orders)}")

if balance["open_margin_calls"]:
    print("🔴 Margin Call : YES")
else:
    print("🟢 Margin Call : None")

print("🟢 Account     : Connected")

print()

line("=")


                          WEBULL PORTFOLIO DASHBOARD                          

⏱️ Updated      : 2026-09-30 10:34:13
🌎 Environment  : SANDBOX / PAPER
💵 Currency     : USD

------------------------------------------------------------------------------
                               ACCOUNT SUMMARY                                
------------------------------------------------------------------------------
💰 Net Liquidation Value : $1,000,000.00
💵 Cash Balance          : $1,000,000.00
📊 Market Value          : $0.00
📈 Unrealized P/L        : +$0.00
📅 Day P/L               : +$0.00
🛡️ Maintenance Margin    : $0.00

------------------------------------------------------------------------------
                                  PORTFOLIO                                   
------------------------------------------------------------------------------
Symbol         Qty         Price      Market Value           P/L
---------------------------------------------------------------------------

In [21]:
# ============================================================
# TRADE HISTORY
# ============================================================

import csv
import os

print()
line()
print("TRADE HISTORY".center(78))
line()

orders = bridge.get_order_history()

# ============================================================
# เตรียม CSV
# ============================================================

# ระบุ Algorithm และ Scheme ที่กำลังรัน
ALGORITHM = "ppo"
SCHEME = "expanding_window"

# สร้าง path:
# trade_data/
# └── ppo/
#     └── expanding_window/
#         └── trade_history.csv

CSV_FILE = os.path.join(
    "trade_data",
    ALGORITHM,
    SCHEME,
    "trade_history.csv"
)

# สร้างโฟลเดอร์อัตโนมัติถ้ายังไม่มี
os.makedirs(os.path.dirname(CSV_FILE), exist_ok=True)


# ============================================================
# อ่าน Order ID ที่มีอยู่แล้วใน CSV
# ============================================================

existing_order_ids = set()

if os.path.exists(CSV_FILE):

    with open(
        CSV_FILE,
        "r",
        newline="",
        encoding="utf-8-sig"
    ) as f:

        reader = csv.DictReader(f)

        for row in reader:

            order_id = row.get("Order_ID")

            if order_id:
                existing_order_ids.add(order_id)


# ============================================================
# เปิด CSV แบบ append
# ============================================================

file_exists = os.path.exists(CSV_FILE)

with open(
    CSV_FILE,
    "a",
    newline="",
    encoding="utf-8-sig"
) as f:

    writer = csv.writer(f)

    # ========================================================
    # ถ้าเป็นไฟล์ใหม่ → สร้าง Header
    # ========================================================

    if not file_exists:

        writer.writerow([
            "Order_ID",
            "Date",
            "Time",
            "Symbol",
            "Side",
            "Qty",
            "Price",
            "Status"
        ])


    # ========================================================
    # Terminal Header
    # ========================================================

    print(
        f"{'Date':<12}"
        f"{'Time':<10}"
        f"{'Symbol':<8}"
        f"{'Side':<8}"
        f"{'Qty':>8}"
        f"{'Price':>14}"
        f"{'Status':>15}"
    )

    line()


    # ========================================================
    # อ่าน Trade History
    # ========================================================

    for order in orders:

        for trade in order.get("orders", []):

            # ------------------------------------------------
            # Order ID
            # ------------------------------------------------

            order_id = trade.get("order_id", "-")


            # ------------------------------------------------
            # วันที่ / เวลา
            # ------------------------------------------------

            datetime_str = trade.get(
                "filled_time_at",
                "-"
            )

            if datetime_str != "-":

                date = datetime_str[:10]
                time = datetime_str[11:19]

            else:

                date = "-"
                time = "-"


            # ------------------------------------------------
            # ข้อมูล Trade
            # ------------------------------------------------

            symbol = trade.get("symbol", "-")
            side = trade.get("side", "-")
            quantity = trade.get("filled_quantity", "0")
            price = trade.get("filled_price", "0")
            status = trade.get("status", "-")


            # =================================================
            # แสดงใน Terminal
            # =================================================

            print(
                f"{date:<12}"
                f"{time:<10}"
                f"{symbol:<8}"
                f"{side:<8}"
                f"{int(float(quantity)):>8,}"
                f"{money(price):>14}"
                f"{status:>15}"
            )


            # =================================================
            # บันทึก CSV เฉพาะ Order ใหม่
            # =================================================

            if order_id not in existing_order_ids:

                writer.writerow([
                    order_id,
                    date,
                    time,
                    symbol,
                    side,
                    int(float(quantity)),
                    price,
                    status
                ])

                # เพิ่มเข้า set
                # เพื่อป้องกันการบันทึกซ้ำ
                existing_order_ids.add(order_id)


print()
line("=")

print(f"📁 Trade history saved to: {CSV_FILE}")


------------------------------------------------------------------------------
                                TRADE HISTORY                                 
------------------------------------------------------------------------------
Date        Time      Symbol  Side         Qty         Price         Status
------------------------------------------------------------------------------
2026-09-29  16:01:42  TLT     BUY        2,551        $77.96         FILLED
2026-09-29  16:01:40  SPY     BUY          261       $763.11         FILLED
2026-09-29  16:01:40  QQQ     BUY          271       $737.10         FILLED
2026-09-29  16:01:41  GLD     BUY          526       $381.04         FILLED
2026-09-29  16:01:41  DIA     BUY          392       $510.71         FILLED

📁 Trade history saved to: trade_data\ppo\expanding_window\trade_history.csv
